# Main Figure Default Bloch Sphere

## Imports

In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate();
# using Piccolo
using PiccoloQuantumObjects
using QuantumCollocation
using ForwardDiff
using LinearAlgebra
# using Plots
using SparseArrays
using Statistics
using CairoMakie
using Random
using NamedTrajectories
using LaTeXStrings
using GeometryBasics

  Activating project at `~/notebooks/src`
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ Piccolissimo
│  └─ QuantumCollocation
└ @ Base.Precompilation precompilation.jl:651
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ Piccolissimo
│  └─ QuantumCollocation
└ @ Base.Precompilation precompilation.jl:651
┌ Warning: Replacing docs for `QuantumCollocation.ProblemTemplates.UnitaryUniversalProblem :: Union{}` in module `QuantumCollocation.ProblemTemplates`
└ @ Base.Docs docs/Docs.jl:243
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Warning: Replacing docs for `QuantumCollocation.ProblemTemplates.UnitaryUniversalProblem :: Union{}` in module `QuantumCollocation.ProblemTemplates`
└ @ Base.Docs docs/Docs.jl:243


In [2]:
# Problem parameters
T = 40
Δt = 0.8
U_goal = GATES.H
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]
piccolo_opts = PiccoloOptions(verbose=false)
pretty_print(X::AbstractMatrix) = Base.show(stdout, "text/plain", X);
sys = QuantumSystem(H_drive)
seed = 46#rand(1:1000, 1)
F=0.9999
num_iter = 6000
hess = false
hess_iter = 120
Q = 10
a_bound = 1.0
dda_bound = 0.5

0.5

In [3]:
# Adjoint, rollout initialization
Random.seed!(seed)
∂ₑHₐ = [PAULIS.Z]
varsys_add = VariationalQuantumSystem(
    H_drive,
    ∂ₑHₐ
)

prob = UnitarySmoothPulseProblem(
    sys, U_goal, T, Δt; Δt_min=Δt, Δt_max=Δt, a_bound=a_bound, dda_bound=dda_bound
)
        
solve!(prob, max_iter=num_iter, print_level=5, options=IpoptOptions(eval_hessian=false))

    constructing UnitarySmoothPulseProblem...
	using integrator: DataType
	control derivative names: [:da, :dda]
	applying timesteps_all_equal constraint: Δt
    initializing optimizer...
        applying constraint: timesteps all equal constraint
        applying constraint: initial value of Ũ⃗
        applying constraint: initial value of a
        applying constraint: final value of a
        applying constraint: bounds on a
        applying constraint: bounds on da
        applying constraint: bounds on dda
        applying constraint: bounds on Δt

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear sol

In [4]:
# Bloch Data
x1 = 0.005
x2 = 0.05
steps = 10
step_size = (x2 - x1) / steps
epsilons = x1:step_size:x2

colors = Makie.wong_colors()
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]
script_e= latexstring("\\mathcal{E}")
ket_0 = [1.0,0.0]
rho_0 = ket_0 * ket_0'
def_traj = prob.trajectory
T = def_traj.T

# exectation values for trajectory
expect_val_x = [real(tr(PAULIS.X * iso_vec_to_operator(def_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(def_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y = [real(tr(PAULIS.Y * iso_vec_to_operator(def_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(def_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z = [real(tr(PAULIS.Z * iso_vec_to_operator(def_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(def_traj.Ũ⃗[:, t])')) for t in 1:T]

# Store trajectories for perturbed systems
def_perturbed_trajectories = []
for eps in epsilons
    # Create perturbed system
    perturbed_sys = QuantumSystem(eps*PAULIS.Z, H_drive)
    Ũ⃗_var = unitary_rollout(def_traj, perturbed_sys)

    # Calculate expectation values for perturbed trajectory
    def_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    def_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    def_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    
    push!(def_perturbed_trajectories, (def_exp_x, def_exp_y, def_exp_z, eps))
end

In [ ]:
# Plotting
f  = CairoMakie.Figure(resolution = (800, 600), figure_padding=0)
ax = CairoMakie.Axis3(f[1, 1];
    aspect = :equal,
    xspinesvisible = false,
    yspinesvisible = false,
    zspinesvisible = false,
    xgridvisible  = false,
    ygridvisible  = false,
    zgridvisible  = false,
    xypanelvisible = false,
    xzpanelvisible = false,
    yzpanelvisible = false,
)

palette = to_colormap(:tab10)
styles  = (:solid, :dash, :dot, :dashdot)

origins = [Point3f(0,0,0), Point3f(0,0,0), Point3f(0,0,0)]
dirs    = [Vec3f(1.0,0,0), Vec3f(0,1.0,0), Vec3f(0,0,1.0)]

CairoMakie.arrows!(ax, origins, dirs;
    color = [:red, :green, :blue],
    arrowsize = 0.05,
    linewidth = 0.01
)

CairoMakie.text!(ax, "x", position = Point3f(1.2, -0.15, 0), align = (:left, :center),  color = :red,   fontsize = 28)
CairoMakie.text!(ax, "y", position = Point3f(0, 1.15, 0), align = (:center, :bottom), color = :green, fontsize = 28)
CairoMakie.text!(ax, "z", position = Point3f(0, 0, 1.1), align = (:center, :bottom), color = :blue,  fontsize = 28)

# Plot original trajectories
CairoMakie.lines!(ax, real.(expect_val_x), real.(expect_val_y), real.(expect_val_z);
    color     = :black,
    linestyle = styles[1],
    label     = latexstring("|0⟩ \\text{ to } |+⟩, \\; \\mathcal{E}"),
    linewidth = 1.5
)


# Plot perturbed trajectories with fading colors
for (i, (def_exp_x, def_exp_y, def_exp_z, eps)) in enumerate(def_perturbed_trajectories)
    # Calculate alpha (transparency) based on epsilon value
    # Higher epsilon = more transparent (fainter)
    alpha = 1.0 - (i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
    # Use a different color from the palette and adjust with RGBA
    color_with_alpha = (:black, alpha)
    
    CairoMakie.lines!(ax, real.(def_exp_x), real.(def_exp_y), real.(def_exp_z);
        color     = color_with_alpha,
        linestyle = styles[1],
        # label     = "Perturbed ε=$(eps)",
        linewidth = 2.0 - 0.3*i  # Also make lines thinner as epsilon increases
    )
    
    # Add endpoint markers for perturbed trajectories
    CairoMakie.scatter!(ax, [real(def_exp_x[end])], [real(def_exp_y[end])], [real(def_exp_z[end])];
        color = color_with_alpha, 
        markersize = 2.0 - 0.3*i,  # Smaller markers for higher epsilon
        marker = :diamond
    )
end

# Original endpoint markers
CairoMakie.scatter!(ax, [real(expect_val_x[end])], [real(expect_val_y[end])], [real(expect_val_z[end])];
    color = :black, markersize = 20, marker = :xcross)

CairoMakie.mesh!(ax, Sphere(Point3f(0,0,0), 1f0);
    color = (0.2, 0.6, 1.0, 0.05),
    transparency = true,
    shading = true
)

CairoMakie.xlims!(ax, -1.3, 1.3)
CairoMakie.ylims!(ax, -1.3, 1.3)
CairoMakie.zlims!(ax, -1.3, 1.3)

ax.azimuth[]   =  π/20
ax.elevation[] =  π/20


#CairoMakie.axislegend(ax; position = :lt)
CairoMakie.hidexdecorations!(ax, grid = true)
CairoMakie.hideydecorations!(ax, grid = true)
CairoMakie.hidezdecorations!(ax, grid = true)

f

base_path = "./artifacts/plots"
new_folder_name = "main_fig_plots" # <-- This is the new folder I named
filename = "def_bloch.png" # You can use .png, .pdf, or .svg

# Create the full directory path
save_directory = joinpath(base_path, new_folder_name)

# Create the directory (and any intermediate paths) if it doesn't exist
# mkpath! doesn't error if the folder already exists
mkpath(save_directory)

# Define the full path for the file
full_save_path = joinpath(save_directory, filename)

# Save the figure
# This replaces the old last line ("fig")
# It will save the file and also display the fig in a notebook/REPL
save(full_save_path, f, px_per_unit = 300)

display(f)

In [6]:
# this cell saves the trajectory data and metadata

# using JLD2, FileIO

# # Create the directory
# artifacts_dir = "artifacts/def_bloch_data"
# mkpath(artifacts_dir)

# # Save htog_probs data for each seed in separate files
# for (i, seed) in enumerate(seeds)
#     save(joinpath(artifacts_dir, "prob.jld2"), Dict(
#         "prob" => prob,
#         "seed" => seed,
#         "F" => F,
#         "num_iter" => num_iter,
#         "hess_iter" => hess_iter,
#         "a_bound" => a_bound,
#         "dda_bound" => dda_bound,
#     ))
# end

# # Save the figure
# save(joinpath(artifacts_dir, "plot.png"), fig)

# println("Data saved to $artifacts_dir")
# println("Created $(length(seeds)) separate files, one for each seed")